要运行此程序，请在**免费** Tesla T4 Google Colab 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  #  Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### 不懒惰

In [ ]:
from unsloth import FastSentenceTransformer

fourbit_models = [
    "unsloth/all-MiniLM-L6-v2",
    "unsloth/embeddinggemma-300m",
    "unsloth/Qwen3-Embedding-4B",
    "unsloth/Qwen3-Embedding-0.6B",
    "unsloth/all-mpnet-base-v2",
    "unsloth/gte-modernbert-base",
    "unsloth/bge-m3"

] # 更多模型请访问 https://huggingface.co/unsloth

model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/embeddinggemma-300m",
    max_seq_length = 1024,   # 选择任何一个用于长上下文！
    full_finetuning = False, # [新！] 我们现在进行了全面的微调！
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.8: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


我们现在添加了 LoRA 适配器，因此我们只需要更新少量参数！

In [2]:
model = FastSentenceTransformer.get_peft_model(
    model,
    r = 32, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # 支持任意，但 = 0 已优化
    bias = "none",    # 支持任何，但=“无”已优化
    # [新]“unsloth”使用的 VRAM 减少了 30%，适合 2 倍大的批量大小！
    use_gradient_checkpointing = "unsloth", # 对于很长的上下文来说是真实的或“不懒惰的”
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
    task_type = "FEATURE_EXTRACTION"
)

Unsloth: Making `model.base_model.model` require gradients


<a名称=“数据”></a>
### 数据准备
我们现在使用“tomaarsen/miriad-4.4M-split”数据集，这是从同行评审的生物医学文献中提取的 440 万个医学问答对的大规模集合。为了保持效率，我们使用数据流来摄取 10,000 个训练样本和 2,000 个评估样本的子集。

In [3]:
from datasets import load_dataset,Dataset

stream_train = list(load_dataset("tomaarsen/miriad-4.4M-split", split = "train",streaming = True).take(10000))
stream_eval = list(load_dataset("tomaarsen/miriad-4.4M-split", split = "eval",streaming = True).take(2000))

train_dataset = Dataset.from_generator(lambda: (yield from stream_train))
eval_dataset = Dataset.from_generator(lambda: (yield from stream_eval))

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

我们看一下数据集结构：

In [4]:
train_dataset[0]

{'question': 'What factors may contribute to increased pulmonary conduit durability in patients who undergo the Ross operation compared to those with right ventricular outflow tract obstruction?\n',
 'passage_text': "I n 1966, Ross and Somerville 1 reported the first use of an aortic homograft to establish right ventricle-to-pulmonary artery continuity in a patient with tetralogy of Fallot and pulmonary atresia. Since that time, pulmonary position homografts have been used in a variety of right-sided congenital heart lesions. Actuarial 5-year homograft survivals for cryopreserved homografts are reported to range between 55% and 94%, with the shortest durability noted in patients less than 2 years of age. 4 Pulmonary position homografts also are used to replace pulmonary autografts explanted to repair left-sided outflow disease (the Ross operation). Several factors may be likely to favor increased pulmonary conduit durability in Ross patients compared with those with right ventricular o

## 基准性能
### 现在微调后让我们评估模型！

In [5]:
import torch
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries = dict(enumerate(eval_dataset["question"]))
corpus = dict(enumerate(list(eval_dataset["passage_text"]) + train_dataset["passage_text"][:2000]))
relevant_docs = {idx: [idx] for idx in queries}
evaluator = InformationRetrievalEvaluator(
    queries = queries,
    corpus = corpus,
    relevant_docs = relevant_docs,
    show_progress_bar = False,
    batch_size = 64,
)
with torch.autocast(device_type = "cuda", dtype = model.dtype, enabled = model.dtype != torch.float16):
    print(evaluator(model))

{'cosine_accuracy@1': 0.8715,
 'cosine_accuracy@3': 0.947,
 'cosine_accuracy@5': 0.9605,
 'cosine_accuracy@10': 0.975,
 'cosine_precision@1': 0.8715,
 'cosine_precision@3': 0.31566666666666665,
 'cosine_precision@5': 0.19210000000000002,
 'cosine_precision@10': 0.0975,
 'cosine_recall@1': 0.8715,
 'cosine_recall@3': 0.947,
 'cosine_recall@5': 0.9605,
 'cosine_recall@10': 0.975,
 'cosine_ndcg@10': 0.926691286916503,
 'cosine_mrr@10': 0.9108168650793647,
 'cosine_map@100': 0.9117156470992975}

<a name="火车"></a>
### 训练模型
现在让我们训练我们的模型。我们使用“MultipleNegativesRankingLoss”

 该损失函数使用同一批次中的其他正例作为负例，这对于对比学习是有效的。

 我们执行 30 个步骤来加快速度，但您可以设置“num_train_epochs=1”以实现完整运行，并关闭“max_steps=None”。

In [6]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported

# 这将使用同一批次中的其他正例作为负例
loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    loss = loss,
    args = SentenceTransformerTrainingArguments(
        # num_train_epochs = 1,
        max_steps = 30,
        per_device_train_batch_size = 64,
        per_device_eval_batch_size = 64,
        gradient_accumulation_steps = 2, # 使用 GA 来模拟批量大小！
        learning_rate = 2e-5,
        logging_steps = 5,
        warmup_ratio = 0.03,
        prompts = {  # 将训练列名称映射到模型提示
          "question": model.prompts["query"],
          "passage_text": model.prompts["document"],
        },
        report_to = "none", # 使用 TrackIO/WandB 等
        eval_strategy = "steps",
        eval_steps = 5,
        bf16 = is_bf16_supported(),
        output_dir = "output",
        lr_scheduler_type = "linear",
        # 因为数据集中有重复的锚点，所以我们不希望
        # 不小心将它们用作反面例子
        batch_sampler = BatchSamplers.NO_DUPLICATES,
    ),
    evaluator = evaluator, # 可选，会使训练速度变慢
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [7]:
# @title 显示当前内存统计信息
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.74 GB of memory reserved.


让我们训练模型吧！要恢复训练运行，请设置“trainer.train(resume_from_checkpoint = True)”

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 2 x 1) = 128
 "-____-"     Trainable parameters = 13,074,432 of 315,937,536 (4.14% trained)

Step,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
5,0.130000,0.092102,0.879000,0.952000,0.962500,0.976500,0.879000,0.317333,0.192500,0.097650,0.879000,0.952000,0.962500,0.976500,0.931557,0.916755,0.917712
10,0.086100,0.072205,0.882000,0.951500,0.964000,0.978000,0.882000,0.317167,0.192800,0.097800,0.882000,0.951500,0.964000,0.978000,0.933386,0.918729,0.919701
15,0.059100,0.064331,0.882500,0.951000,0.965000,0.979000,0.882500,0.317000,0.193000,0.097900,0.882500,0.951000,0.965000,0.979000,0.934128,0.919406,0.920307
20,0.052200,0.059662,0.883500,0.952500,0.966000,0.979500,0.883500,0.317500,0.193200,0.097950,0.883500,0.952500,0.966000,0.979500,0.934963,0.920331,0.921197
25,0.060800,0.056732,0.884000,0.954500,0.966500,0.980000,0.884000,0.318167,0.193300,0.098000,0.884000,0.954500,0.966500,0.980000,0.935508,0.920866,0.921724
30,0.056800,0.055725,0.883000,0.955500,0.967000,0.979500,0.883000,0.318500,0.193400,0.097950,0.883000,0.955500,0.967000,0.979500,0.935071,0.920396,0.921311


In [9]:
# @title 显示最终内存和时间统计数据
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2769.2485 seconds used for training.
46.15 minutes used for training.
Peak reserved memory = 11.287 GB.
Peak reserved memory for training = 3.547 GB.
Peak reserved memory % of max memory = 76.569 %.
Peak reserved memory for training % of max memory = 24.062 %.


### 现在微调后让我们评估模型！

In [15]:
with torch.autocast(device_type = "cuda", dtype = model.dtype, enabled = model.dtype != torch.float16):
    print(evaluator(model))

{'cosine_accuracy@1': 0.883,
 'cosine_accuracy@3': 0.9555,
 'cosine_accuracy@5': 0.967,
 'cosine_accuracy@10': 0.9795,
 'cosine_precision@1': 0.883,
 'cosine_precision@3': 0.31849999999999995,
 'cosine_precision@5': 0.19340000000000004,
 'cosine_precision@10': 0.09795000000000002,
 'cosine_recall@1': 0.883,
 'cosine_recall@3': 0.9555,
 'cosine_recall@5': 0.967,
 'cosine_recall@10': 0.9795,
 'cosine_ndcg@10': 0.9350705550252689,
 'cosine_mrr@10': 0.9203962301587296,
 'cosine_map@100': 0.9213114362986002}

仅 30 个步骤，模型的 Accuracy@1 从 0.871 增加到 0.883，展示了模型适应特定数据集的速度。

<a name="推理"></a>
### 推论
让我们在训练后运行模型来看看改进！

In [22]:
query = "Patient presents with sharp chest pain that improves when leaning forward."

candidates = [
    "Acute Pericarditis often involves pleuritic chest pain relieved by sitting up and leaning forward.",
    "Myocardial Infarction typically presents with crushing substernal pressure and radiation to the left arm.",
    "Pneumothorax is characterized by sudden onset shortness of breath and unilateral chest pain.",
    "Gastroesophageal Reflux Disease (GERD) causes burning retrosternal pain usually after meals."
]

query_emb = model.encode(query, convert_to_tensor = True)
candidate_embs = model.encode(candidates, convert_to_tensor = True)
similarities = model.similarity(query_emb, candidate_embs)

ranking = similarities.argsort(descending = True)[0]

for idx in ranking.tolist():
    score = similarities[0][idx].item()
    text = candidates[idx]
    print(f"{score:.4f} | {text}")

0.7012 | Acute Pericarditis often involves pleuritic chest pain relieved by sitting up and leaning forward.
0.5854 | Pneumothorax is characterized by sudden onset shortness of breath and unilateral chest pain.
0.5420 | Myocardial Infarction typically presents with crushing substernal pressure and radiation to the left arm.
0.4211 | Gastroesophageal Reflux Disease (GERD) causes burning retrosternal pain usually after meals.

<a name="保存"></a>
### 保存、加载微调模型
要将最终模型保存为 LoRA 适配器，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 这仅保存 LoRA 适配器，而不是完整模型。要保存到 16 位，请向下滚动！

In [17]:
model.save_pretrained("embeddinggemma_lora")  # 本地储蓄
model.tokenizer.save_pretrained("embeddinggemma_lora")
# model.push_to_hub("your_name/embeddinggemma_lora", token = "YOUR_HF_TOKEN") # 在线保存
# model.tokenizer.push_to_hub("your_name/embeddinggemma_lora", token = "YOUR_HF_TOKEN") # 在线保存

现在，如果您想加载我们刚刚保存用于推理的 LoRA 适配器，请将“False”设置为“True”：

In [18]:
if False:
    from unsloth import FastSentenceTransformer
    model = FastSentenceTransformer.from_pretrained(
        "model",
    )

### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [19]:
# 合并到16位
if False:
    model.save_pretrained_merged("embeddinggemma_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit",)
if False: # 推送至 HF 集线器
    model.push_to_hub_merged("HF_USERNAME/embeddinggemma_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("embeddinggemma_lora")
if False: # 推送至 HF 集线器
    model.push_to_hub("HF_USERNAME/embeddinggemma_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

In [ ]:
# 保存到8位Q8_0
if False:
    model.save_pretrained_gguf("embeddinggemma_finetune",)
# 记得去 https://huggingface.co/settings/tokens 获取令牌！
# 并将 hf 更改为您的用户名！
if False:
    model.push_to_hub_gguf("HF_USERNAME/embeddinggemma_finetune", token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False:
    model.save_pretrained_gguf("embeddinggemma_finetune", quantization_method = "f16")
if False: # 推送至 HF 集线器
    model.push_to_hub_gguf("HF_USERNAME/embeddinggemma_finetune", quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False:
    model.save_pretrained_gguf("embeddinggemma_finetune", quantization_method = "q4_k_m")
if False: # 推送至 HF 集线器
    model.push_to_hub_gguf("HF_USERNAME/embeddinggemma_finetune", quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# 保存到多个 GGUF 选项 - 如果您想要多个，速度会更快！
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/embeddinggemma_finetune", # 将 hf 更改为您的用户名！
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # 在 https://huggingface.co/settings/tokens 获取令牌
    )

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  <b>此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可</b>
</div>